# 01 - Visualize HILO Dataset (RGB-D)

## Mục tiêu
Hiển thị một số ví dụ từ dataset HILO: ảnh RGB, ảnh depth dạng heatmap, và (nếu có) point cloud, phục vụ việc chèn hình vào báo cáo.

---

Trong notebook này:
- Chọn một vài mẫu ảnh từ dataset
- Hiển thị RGB và Depth cạnh nhau
- Overlay bounding box (nếu có annotation)
- Chuẩn bị các hình sẵn để chụp màn hình cho báo cáo.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import cv2
from pathlib import Path

# Matplotlib style
plt.style.use("seaborn-v0_8-darkgrid")
plt.rcParams["figure.figsize"] = (12, 8)
plt.rcParams["font.size"] = 12
plt.rcParams["axes.titlesize"] = 14
plt.rcParams["axes.labelsize"] = 12

# Các candidate path cho dữ liệu
PROJECT_ROOT = Path("D:/multi_modal_robot_ai").resolve()

HILO_DIRS = [
    PROJECT_ROOT / "data" / "HILO",                 # như mô tả trong prompt
    PROJECT_ROOT / "data" / "03_external" / "HILO",  # theo cấu trúc hiện tại
]
SAMPLES_DIR = PROJECT_ROOT / "data" / "samples"

print("PROJECT_ROOT =", PROJECT_ROOT)
print("HILO candidates:")
for d in HILO_DIRS:
    print(" -", d, "=>", "OK" if d.exists() else "missing")
print("SAMPLES_DIR =", SAMPLES_DIR, "=>", "OK" if SAMPLES_DIR.exists() else "missing")


PROJECT_ROOT = D:\multi_modal_robot_ai
HILO candidates:
 - D:\multi_modal_robot_ai\data\HILO => missing
 - D:\multi_modal_robot_ai\data\03_external\HILO => OK
SAMPLES_DIR = D:\multi_modal_robot_ai\data\samples => missing


In [ ]:
def find_sample_images(max_images: int = 4):
    """Tìm một vài ảnh RGB/Depth trong dataset hoặc thư mục samples."""
    rgb_paths = []
    depth_paths = []

    # Ưu tiên thư mục samples nếu tồn tại
    search_dirs = []
    if SAMPLES_DIR.exists():
        search_dirs.append(SAMPLES_DIR)
    for d in HILO_DIRS:
        if d.exists():
            search_dirs.append(d)

    exts = ["*.png", "*.jpg", "*.jpeg"]

    for d in search_dirs:
        for ext in exts:
            for p in d.rglob(ext):
                # heuristic đơn giản: file chứa "depth" hoặc "d_" coi là depth
                if "depth" in p.stem.lower() or p.stem.lower().startswith("d_"):
                    depth_paths.append(p)
                else:
                    rgb_paths.append(p)
                if len(rgb_paths) >= max_images and len(depth_paths) >= max_images:
                    return rgb_paths, depth_paths

    return rgb_paths, depth_paths

rgb_paths, depth_paths = find_sample_images(max_images=4)
print(f"Tìm được {len(rgb_paths)} ảnh RGB, {len(depth_paths)} ảnh depth")
if not rgb_paths:
    print("⚠️ Không tìm thấy ảnh RGB. Hãy kiểm tra lại đường dẫn dataset.")
if not depth_paths:
    print("⚠️ Không tìm thấy ảnh Depth. Sẽ chỉ hiển thị RGB nếu có.")


Tìm được 0 ảnh RGB, 0 ảnh depth
⚠️ Không tìm thấy ảnh RGB. Hãy kiểm tra lại đường dẫn dataset.
⚠️ Không tìm thấy ảnh Depth. Sẽ chỉ hiển thị RGB nếu có.


In [ ]:
def show_rgb_depth_pairs(n_show: int = 3):
    n = min(n_show, len(rgb_paths))
    if n == 0:
        print("Không có ảnh để hiển thị.")
        return

    for i in range(n):
        rgb_path = rgb_paths[i]
        depth_path = depth_paths[i] if i < len(depth_paths) else None

        rgb = cv2.imread(str(rgb_path))
        if rgb is None:
            continue
        rgb = cv2.cvtColor(rgb, cv2.COLOR_BGR2RGB)

        fig, axes = plt.subplots(1, 2 if depth_path else 1, figsize=(12, 5))
        if not isinstance(axes, np.ndarray):
            axes = np.array([axes])

        axes[0].imshow(rgb)
        axes[0].set_title(f"RGB: {rgb_path.name}")
        axes[0].axis("off")

        if depth_path is not None:
            depth = cv2.imread(str(depth_path), cv2.IMREAD_UNCHANGED)
            if depth is not None:
                depth_norm = cv2.normalize(depth, None, 0, 255, cv2.NORM_MINMAX)
                depth_color = cv2.applyColorMap(depth_norm.astype(np.uint8), cv2.COLORMAP_PLASMA)
                depth_rgb = cv2.cvtColor(depth_color, cv2.COLOR_BGR2RGB)
                axes[1].imshow(depth_rgb)
                axes[1].set_title(f"Depth (heatmap): {depth_path.name}")
                axes[1].axis("off")

        plt.suptitle("Ví dụ RGB-D từ dataset HILO / samples", fontsize=14)
        plt.tight_layout()
        plt.show()

show_rgb_depth_pairs(n_show=3)


Không có ảnh để hiển thị.


## Ghi chú cho báo cáo

- Hình trên minh hoạ **ảnh RGB** và **Depth heatmap** từ dataset.
- Có thể chụp màn hình từng figure và chèn vào các mục:
  - "Mô tả dữ liệu đầu vào của hệ thống"
  - "Ví dụ RGB-D từ camera / dataset HILO".

Nếu cần point cloud 3D:
- Có thể dùng `open3d` và lưu hình riêng, sau đó chèn thêm vào notebook hoặc trực tiếp vào báo cáo.
